In [4]:
import os
import glob
import pickle
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar

root = "/kaggle/input/datasets/subihanbiswas/pkl-files"

run_files = sorted(glob.glob(os.path.join(root, "raw_predictions_*.pkl")))

print(f"Pooling across {len(run_files)} runs")
print(*run_files, sep="\n")

Pooling across 4 runs
/kaggle/input/datasets/subihanbiswas/pkl-files/raw_predictions_original_run.pkl
/kaggle/input/datasets/subihanbiswas/pkl-files/raw_predictions_seed101.pkl
/kaggle/input/datasets/subihanbiswas/pkl-files/raw_predictions_seed202.pkl
/kaggle/input/datasets/subihanbiswas/pkl-files/raw_predictions_seed303.pkl


In [6]:
degradation_conditions = {
    "All present":        None,
    "Fingerprint missing": ["fp"],
    "Iris missing":        ["iris"],
    "Voice missing":       ["voice"],
    "FP + Iris missing":   ["fp", "iris"],
    "FP + Voice missing":  ["fp", "voice"],
    "Iris + Voice missing":["iris", "voice"],
}


In [7]:
all_runs = [pickle.load(open(f, "rb")) for f in run_files]

print(f"\n{'Condition':25s} {'n_pooled':>10s} {'p-value':>10s}   Interpretation")
print("-" * 75)

for cond in degradation_conditions:
    pooled_adaptive_preds = np.concatenate([r["adaptive"][cond]["preds"] for r in all_runs])
    pooled_adaptive_labels = np.concatenate([r["adaptive"][cond]["labels"] for r in all_runs])
    pooled_naive_preds = np.concatenate([r["naive"][cond]["preds"] for r in all_runs])
    pooled_naive_labels = np.concatenate([r["naive"][cond]["labels"] for r in all_runs])

    assert np.array_equal(pooled_adaptive_labels, pooled_naive_labels)

    a_correct = (pooled_adaptive_preds == pooled_adaptive_labels)
    n_correct = (pooled_naive_preds == pooled_naive_labels)

    both_correct = int(np.sum(a_correct & n_correct))
    only_adaptive = int(np.sum(a_correct & ~n_correct))
    only_naive = int(np.sum(~a_correct & n_correct))
    both_wrong = int(np.sum(~a_correct & ~n_correct))

    table = [[both_correct, only_adaptive], [only_naive, both_wrong]]
    result = mcnemar(table, exact=(only_adaptive + only_naive < 25))

    sig = "significant (p<0.05)" if result.pvalue < 0.05 else "not significant"
    print(f"{cond:25s} {len(pooled_adaptive_labels):10d} {result.pvalue:10.4f}   {sig}")


Condition                   n_pooled    p-value   Interpretation
---------------------------------------------------------------------------
All present                    13056     0.5000   not significant
Fingerprint missing            13056     0.3750   not significant
Iris missing                   13056     0.2317   not significant
Voice missing                  13056     0.0784   not significant
FP + Iris missing              13056     0.3326   not significant
FP + Voice missing             13056     0.1273   not significant
Iris + Voice missing           13056     0.0488   significant (p<0.05)
